In [5]:
import os
import re
import time
import requests
from bs4 import BeautifulSoup
from urllib.parse import urljoin
from requests.adapters import HTTPAdapter
from urllib3.util.retry import Retry

BASE_URL = "https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/"
OUT_DIR = "uscrn_daily01"

os.makedirs(OUT_DIR, exist_ok=True)

session = requests.Session()
retries = Retry(
    total=5,
    backoff_factor=1,
    status_forcelist=[429, 500, 502, 503, 504],
    allowed_methods=["GET"],
)
session.mount("https://", HTTPAdapter(max_retries=retries))
session.mount("http://", HTTPAdapter(max_retries=retries))


def get_soup(url):
    r = session.get(url, timeout=60)
    r.raise_for_status()
    return BeautifulSoup(r.text, "html.parser")


def get_year_folders(base_url):
    soup = get_soup(base_url)
    year_links = []

    for a in soup.find_all("a"):
        text = a.get_text(strip=True)
        href = a.get("href")

        if re.fullmatch(r"\d{4}/?", text):
            year = text.rstrip("/")
            year_url = urljoin(base_url, year + "/")   
            year_links.append((year, year_url))

    return sorted(year_links, key=lambda x: x[0])


def get_txt_files(year_url):
    soup = get_soup(year_url)
    files = []

    for a in soup.find_all("a"):
        text = a.get_text(strip=True)
        href = a.get("href")

        if href and text.endswith(".txt"):
            file_url = urljoin(year_url, href)   
            files.append((text, file_url))

    return files


def download_file(url, local_path):
    with session.get(url, stream=True, timeout=60) as r:
        r.raise_for_status()
        with open(local_path, "wb") as f:
            for chunk in r.iter_content(chunk_size=1024 * 1024):
                if chunk:
                    f.write(chunk)


year_folders = get_year_folders(BASE_URL)
print(f"Found {len(year_folders)} year folders")

for year, year_url in year_folders:
    year_dir = os.path.join(OUT_DIR, year)
    os.makedirs(year_dir, exist_ok=True)

    print(f"\nProcessing {year}: {year_url}")

    txt_files = get_txt_files(year_url)
    print(f"  Found {len(txt_files)} txt files")

    for fname, file_url in txt_files:
        local_path = os.path.join(year_dir, fname)

        if os.path.exists(local_path):
            continue

        print(f"  Downloading {fname}")
        try:
            download_file(file_url, local_path)
            time.sleep(0.1)
        except Exception as e:
            print(f"  Failed {fname}: {e}")

print("\nDone.")

Found 27 year folders

Processing 2000: https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/2000/
  Found 2 txt files

Processing 2001: https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/2001/
  Found 8 txt files

Processing 2002: https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/2002/
  Found 25 txt files

Processing 2003: https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/2003/
  Found 45 txt files

Processing 2004: https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/2004/
  Found 72 txt files

Processing 2005: https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/2005/
  Found 82 txt files

Processing 2006: https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/2006/
  Found 97 txt files

Processing 2007: https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/2007/
  Found 121 txt files

Processing 2008: https://www.ncei.noaa.gov/pub/data/uscrn/products/daily01/2008/
  Found 137 txt files

Processing 2009: https://www.ncei.noaa.gov/pub/dat